In [2]:
# Project Two Dashboard
# Author: Daniel Zawicki
# Updated: 19 April 2026
# Code pulled from ModuleSixMilestone.ipynb

# Builds an interactive dashboard used to display and filter animals populating a database
# Filters correlate with criteria requirements regarding animal rescue scenarios, e.g. ideal breeds for water, mountain, and disaster rescue
# Dashboard displays map with pin reflecting location of selected animal
# Dashboard displays pie chart reflecting breed counts

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Get the logo to work
import base64

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import AnimalShelter class from CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Updated with username and password and CRUD Python module name. NOTE: You will
# likely need more variables for your constructor to handle the hostname and port of the MongoDB
# server, and the database and collection names

username = "aacuser" # Hard coded username
password = "1234"    # Hard coded password
shelter = AnimalShelter(username, password)


# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# Get the logo to work
image_filename = 'assets/grazioso_salvare_logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Welcome to Daniel Zawicki\'s Unique Geolocator Dashboard!'))), # Unique identifier as dashboard header
    html.Hr(),
    
    html.Div([
        html.H1("Daniel Zawicki's Dashboard"),
        
        html.A([
            # Grazioso Salvare Logo
            html.Img(
                src = 'data:image/png;base64,{}'.format(encoded_image),
                alt = 'Grazioso Salvare Logo',
                style = {'height': '200px'}
            ) 
           ], href = 'https://www.snhu.edu', target = '_blank') # hyperlinked to website
    ], style = {'textAlign': 'center'}),
                              
    html.H4("Daniel Zawicki\'s Radio Items for Selection!"),
    
    # Selectable radio items corresponding with respective query activation
    # Radio items are ideal to ensure only one filter being active at a time
    dcc.RadioItems(
        id = 'filter-type',
        options = [
            {'label': 'Water Rescue', 'value': 'water'},                          # runs water_rescue_query
            {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},      # runs mountain_rescue_query
            {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},    # runs disaster_rescue_query
            {'label': 'Reset', 'value': 'reset'}],                                # clears query
        value = 'reset',                                                          # default radio item
        labelStyle = {'display': 'inline-block', 'margin-right': '20px'}),
    
    html.Hr(),  # horizontal line

    dash_table.DataTable(
        id='datatable-id',
        # Iterate through columns to establish behavior
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        # Set up the features for your interactive data table to make it user-friendly for your client
        editable=False,             # Can NOT edit cells
        filter_action="native",     # Uses Dash's built in behavior
        sort_action="native",       # Uses Dash's built in behavior
        sort_mode="multi",          # Can sort multiple rows
        column_selectable="single", # Can select a single column
        row_selectable="single",    # Can select a single row
        row_deletable=False,        # Can NOT delete rows
        selected_columns=[],        # Default selected column is none
        selected_rows=[0],          # Default selected row is top
        page_action="native",       # Uses Dash's built in behavior
        page_current= 0,            # Default page is first page
        page_size= 10,              # Show 10 rows per page

    ),
    html.Br(),  # line break
    html.Hr(),  # horizontal line
    html.Div([
        # Map
        html.Div(
            id = 'map-id',
            className = 'col s12 m6',
        ),
        # Pie Chart
        html.Div(
            id = 'chart-id',
            className = 'col s12 m6',
        )
    ], style = {'display': 'flex', 'alignItems': 'flex-start'})
])


#############################################
# Querys
# Using $regex for inclusivity e.g. German Shepherd && German Shepherd Mix
#############################################

# query for identifying dogs that meet criteria for water rescue
# corresponding radio item: {'label': 'Water Rescue', 'value': 'water'}
water_rescue_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"sex_upon_outcome": "Intact Female"},
        {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}},  # greater than or equal to // less than or equal to
        {"$or": [
            {"breed": {"$regex": "Labrador Retriever Mix", "$options": "i"}},  # case insensitive, inclusive 
            {"breed": {"$regex": "Chesapeake Bay Retriever", "$options": "i"}},
            {"breed": {"$regex": "Newfoundland", "$options": "i"}}
        ]}
    ]
}

# query for idenifying dogs that meet criteria for mountain rescue
# corresponding radio item: {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'}
mountain_rescue_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"sex_upon_outcome": "Intact Male"},
        {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}},  # greater than or equal to // less than or equal to
        {"$or": [
            {"breed": {"$regex": "German Shepherd", "$options": "i"}},  # case insensitive, inclusive
            {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
            {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
            {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
            {"breed": {"$regex": "Rottweiler", "$options": "i"}}
        ]}
    ]
}

# query for identifying dogs that meet criteria for disaster rescue
# corresponding radio item: {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
disaster_rescue_query = {
    "$and": [
        {"animal_type": "Dog"},
        {"sex_upon_outcome": "Intact Male"},
        {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}},  # greater than or equal to // less than or equal to
        {"$or": [
            {"breed": {"$regex": "Doberman Pinscher", "$options": "i"}},  # case insensitive, inclusive
            {"breed": {"$regex": "German Shepherd", "$options": "i"}},
            {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
            {"breed": {"$regex": "Bloodhound", "$options": "i"}},
            {"breed": {"$regex": "Rottweiler", "$options": "i"}}
        ]}
    ]
}


#############################################
# Interaction Between Components / Controller
#############################################

#This callback controls filter behavior
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')  # when radio button selection changes, the following function runs
)
def update_dashboard(filter_type):
    if filter_type == 'water':
        df = pd.DataFrame.from_records(shelter.read(water_rescue_query))     # water radio item
    elif filter_type == 'mountain':
        df = pd.DataFrame.from_records(shelter.read(mountain_rescue_query))  # mountain radio item
    elif filter_type == 'disaster':
        df = pd.DataFrame.from_records(shelter.read(disaster_rescue_query))  # disaster radio item
    else:
        df = pd.DataFrame.from_records(shelter.read({}))                     # reset radio item, no filter
    
    # Drop _id field for cleaner output
    if '_id' in df.columns:
        df.drop(columns = ['_id'], inplace = True)
            
    return df.to_dict('records')

#This callback will highlight a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),    # Placeholder panel for the map to populate
    [Input('datatable-id', "derived_virtual_data"),    # What data is being displayed
     Input('datatable-id', "derived_virtual_selected_rows")])    # What data the user selected // [0] by default

def update_map(viewData, index):
    # Code for geolocation chart
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can 
    # be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]

# Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
           center=[30.75,-97.48], zoom=10, children=[
           dl.TileLayer(id="base-layer-id"),
           # Marker with tool tip and popup
           # Column 13 and 14 define the grid-coordinates for 
           # the map
           # Column 4 defines the breed for the animal
           # Column 9 defines the name of the animal
           dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
              children=[
              dl.Tooltip(dff.iloc[row,4]),
              dl.Popup([
                html.H1("Animal Name"),
                html.P(dff.iloc[row,9])
             ])
          ])
       ])
    ]

# This callback controls the behavior of the pie chart
@app.callback(
    Output('chart-id','children'),
    Input('datatable-id', 'derived_virtual_data')  # whatever the table is showing, the chart will reflect
)
def update_chart(viewData):
    if viewData is None:  # no filtered data on initial load
        dff = df.copy()
    else: 
        dff = pd.DataFrame.from_dict(viewData)  # use filtered data
        
    if len(dff) == 0:  # if a filter returns no rows of data
        return px.pie(names = [], title = 'Breed Distribution') 
    
    breed_counts = dff['breed'].value_counts().reset_index()  # count how many times each breed appears in data
    breed_counts.columns = ['breed', 'count']  # rename columns appropriately
    
    # build the pie chart
    fig = px.pie(
        breed_counts,
        names = 'breed',
        values = 'count',
        title = 'Breed Distribution'
    )
    
    return [
        dcc.Graph(
            id = 'breed-chart',
            figure = fig
        )
    ]
    
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://aztecdomain-modemtonight-3000.codio.io/proxy/8050/
